In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:

extreme_returns = pd.read_csv("data/sp500_pct_extremes.csv")

extreme_returns

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,previous_adj_close,arithmetic_return,arithmetic_return_%,log_return,index,Dividends,Stock Splits,z_score_global,extreme
0,2020-03-09,APA,13.420000,13.700000,9.320000,9.550000,8.224354,28073200.0,17.826614,-0.538647,-53.864746,-0.773593,NaN,NaN,NaN,-39.923853,True
1,2024-04-11,GL,98.480003,98.580002,38.950001,49.169998,48.298512,36577500.0,103.070229,-0.531402,-53.140192,-0.758010,NaN,NaN,NaN,-39.120169,True
2,2019-01-14,PCG,9.210000,9.730000,7.780000,8.380000,8.260254,127198800.0,17.338648,-0.523593,-52.359296,-0.741483,NaN,NaN,NaN,-38.267788,True
3,2020-03-09,OXY,15.580000,19.190001,12.040000,12.510000,11.708246,104930300.0,24.399191,-0.520138,-52.013793,-0.734257,35302.0,0.79,0.0,-37.895106,True
4,2025-10-29,FISV,71.360001,76.650002,66.580002,70.599998,70.599998,103454200.0,126.169998,-0.440438,-44.043751,-0.580600,NaN,NaN,NaN,-29.970347,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728,2025-04-04,BKR,39.290001,39.680000,35.279999,35.410000,34.706951,20974600.0,40.048748,-0.133382,-13.338237,-0.143157,NaN,NaN,NaN,-7.409455,True
1729,2018-11-16,EIX,52.860001,55.009998,52.509998,54.450001,38.770332,8828700.0,33.600945,0.153847,15.384650,0.143101,NaN,NaN,NaN,7.354191,True
1730,2020-03-16,EIX,48.470001,51.110001,43.630001,46.150002,34.412170,4654000.0,39.706360,-0.133334,-13.333354,-0.143101,NaN,NaN,NaN,-7.406549,True
1731,2020-03-17,ATO,96.199997,108.330002,95.790001,108.089996,93.097412,1887700.0,80.686157,0.153821,15.382137,0.143079,NaN,NaN,NaN,7.353067,True


In [5]:
# -----------------------------
# 1. Split tails
# -----------------------------
positive_tail = extreme_returns[
    extreme_returns["arithmetic_return"] > 0
]["arithmetic_return"]
negative_tail = extreme_returns[
    extreme_returns["arithmetic_return"] < 0
]["arithmetic_return"]

# Fix: use absolute values for comparison
negative_tail_abs = negative_tail.abs()

# -----------------------------
# 2. Basic stats
# -----------------------------
print("Positive tail:"); print(positive_tail.describe(), "\n")
print("Negative tail (abs):"); print(negative_tail_abs.describe(), "\n")

# -----------------------------
# 3. Welch t-test — same mean magnitude?
# -----------------------------
t_stat, p_value = stats.ttest_ind(
    positive_tail, negative_tail_abs, equal_var=False
)
print(f"Welch t-test:  t = {t_stat:.4f}, p = {p_value:.6f}\n")

# -----------------------------
# 4. F-test — same variance?
# -----------------------------
lev_stat, lev_p = stats.levene(positive_tail, negative_tail_abs)
print(f"F test: W = {lev_stat:.4f}, p = {lev_p:.6f}\n")


Positive tail:
count    699.000000
mean       0.203033
std        0.064469
min        0.153741
25%        0.166180
50%        0.181980
75%        0.218501
max        0.745932
Name: arithmetic_return, dtype: float64 

Negative tail (abs):
count    1034.000000
mean        0.177270
std         0.051389
min         0.133334
25%         0.144931
50%         0.161352
75%         0.190170
max         0.538647
Name: arithmetic_return, dtype: float64 

Welch t-test:  t = 8.8367, p = 0.000000

F test: W = 4.5316, p = 0.033415



So, statistically, the mean distribution of both tail do not seem to be statistically equal.

With the count we can say that crashes happen more often but rallies are more violent.